# Notebook 04 — CMF-Static Unlearning (Paper Protocol)

**Paper:** *An Illusion of Unlearning? Assessing Machine Unlearning Through Internal Representations*  
(Gao, Unal, Rangamani, Zhu — AISTATS 2026 · arXiv:2604.08271v1)

**Protocol & Overview:**
- Loads `theta_o_state` from NB1 into a proper `ModelModule` (with CMF head).
- Runs `run_cmf_static()` (paper-faithful Algorithm 2: per-epoch recompute_cmf → freeze W → update encoder) for all 5 base methods:
  `grad_ascent_descent`, `random_label`, `salun`, `scrub`, `tarun`.
- Hyperparameters exclusively from Table 4 (`paper_hparams.py`).
- Slices forget/retain classes across seeds / classes, records Output / Probe / NCC metrics.
- Saves self-describing checkpoints: `{base_method}_cmf_static_{mean_source}_class{forget_class}_seed{seed}.pt`.

In [1]:
import subprocess, sys
def sh(cmd, verbose=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if verbose and r.stdout: print(r.stdout[-4000:])
    if r.returncode != 0 and r.stderr: print('STDERR:', r.stderr[-2000:])
    return r.returncode
sh('pip install -q timm einops scikit-learn matplotlib seaborn pytorch-lightning torchmetrics')

0

In [2]:
import os, sys, json, random, math, time, copy, traceback
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib; import matplotlib.pyplot as plt
matplotlib.rcParams.update({'figure.dpi': 110})
print('PyTorch:', torch.__version__, '  CUDA:', torch.cuda.is_available())

PyTorch: 2.10.0+cu128   CUDA: True


In [3]:
REPO_DIR = '/kaggle/working/CMF_Unlearning'
if not os.path.isdir(REPO_DIR):
    sh(f'git clone https://github.com/tiensinh2/CMF_Unlearning.git {REPO_DIR}')
else:
    sh(f'git -C {REPO_DIR} remote set-url origin https://github.com/tiensinh2/CMF_Unlearning.git')
    sh(f'git -C {REPO_DIR} pull origin main')
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
result = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'],
                        capture_output=True, text=True)
REPO_COMMIT = result.stdout.strip() or 'main'
print('Repo commit:', REPO_COMMIT)

Repo commit: 532c642deae6b1d27e88d0ae6a7070b8264a1ad2


In [4]:
CKPT_DATASET_DIR = '/kaggle/input/datasets/btk23021592/cmf-notebook1'
_CONFIG_CANDIDATES = [
    f'{CKPT_DATASET_DIR}/cmf_benchmark_config.json',
    f'{CKPT_DATASET_DIR}/checkpoints/cmf_benchmark/cmf_benchmark_config.json',
    f'{CKPT_DATASET_DIR}/cmf_benchmark/cmf_benchmark_config.json',
    './notebooks/Result_nb1/checkpoints/regun/regun_config.json',
]
config_path = CKPT_ROOT_NB1 = None
for _p in _CONFIG_CANDIDATES:
    if os.path.exists(_p):
        config_path = _p; CKPT_ROOT_NB1 = os.path.dirname(_p); break

if config_path and os.path.exists(config_path):
    with open(config_path) as f: NB1_CFG = json.load(f)
    DATASET     = NB1_CFG.get('dataset', 'cifar10')
    ARCH        = NB1_CFG.get('arch', 'resnet18')
    NUM_CLASSES = NB1_CFG.get('num_classes', 10)
    TEST_MODE   = NB1_CFG.get('test_mode', False)
else:
    DATASET     = 'cifar10'
    ARCH        = 'resnet18'
    NUM_CLASSES = 10
    TEST_MODE   = False
    CKPT_ROOT_NB1 = '/kaggle/working/checkpoints/cmf_benchmark'

STAGE = '4a'
FORGET_CLASSES = list(range(NUM_CLASSES))
SEEDS          = [0]
THETA_O_SEED   = 0
BASE_METHODS   = ['grad_ascent_descent', 'random_label', 'salun', 'scrub', 'tarun']
MEAN_SOURCES   = ['train']

# Hyperparameters from Table 4 (paper_hparams.py)
CMF_EPOCHS_BY_METHOD = {
    'random_label':        1 if TEST_MODE else 4,
    'salun':               1 if TEST_MODE else 4,
    'grad_ascent_descent': 1 if TEST_MODE else 3,
    'scrub':               1 if TEST_MODE else 3,
    'tarun':               1 if TEST_MODE else 3,
}
CMF_LR = {
    'random_label':        2e-3,
    'salun':               2e-3,
    'grad_ascent_descent': 1e-4,
    'scrub':               5e-3,
    'tarun':               5e-5,
}
CMF_BATCH = {
    'scrub': 64,
}
CMF_BATCH_DEFAULT = 128

CKPT_ROOT = '/kaggle/working/checkpoints/cmf_paper'
os.makedirs(f'{CKPT_ROOT}/cmf_static', exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'STAGE={STAGE}  DATASET={DATASET}  ARCH={ARCH}  device={device}')
print(f'CMF_EPOCHS_BY_METHOD={CMF_EPOCHS_BY_METHOD}')

STAGE=4a  DATASET=cifar10  ARCH=resnet18  device=cuda
CMF_EPOCHS_BY_METHOD={'random_label': 4, 'salun': 4, 'grad_ascent_descent': 3, 'scrub': 3, 'tarun': 3}


In [5]:
import torchvision, torchvision.transforms as transforms
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

full_train      = torchvision.datasets.CIFAR10('/kaggle/working/data', train=True,
                                               download=True,  transform=transform_train)
full_train_eval = torchvision.datasets.CIFAR10('/kaggle/working/data', train=True,
                                               download=False, transform=transform_test)
test_set        = torchvision.datasets.CIFAR10('/kaggle/working/data', train=False,
                                               download=True, transform=transform_test)

# Pre-index by class label
test_targets = torch.tensor(test_set.targets)
TEST_CLASS_IDX = {
    c: (test_targets == c).nonzero(as_tuple=True)[0].tolist()
    for c in range(NUM_CLASSES)
}
train_targets = torch.tensor(full_train.targets)
TRAIN_CLASS_IDX = {
    c: (train_targets == c).nonzero(as_tuple=True)[0].tolist()
    for c in range(NUM_CLASSES)
}
print(f'Train size: {len(full_train)}  Test size: {len(test_set)}')

100%|██████████| 170M/170M [45:13<00:00, 62.8kB/s]


Train size: 50000  Test size: 10000


In [6]:
import argparse
from unlearn.cmf_weights import ModelModule
from unlearn.cmf_two_stage import run_cmf_static

def build_cmf_model(args):
    """Construct proper ModelModule with CMF classifier head."""
    model = ModelModule(args).to(device)
    assert hasattr(model, 'CMFweights'), 'Model must have CMFweights attribute!'
    return model

def make_cmf_args(base_method, lr, epochs, mean_source, forget_class,
                  forget_train_idx, retain_train_idx, seed=0):
    return argparse.Namespace(
        dataset=DATASET, arch=ARCH, num_classes=NUM_CLASSES,
        class_label_names=list(range(NUM_CLASSES)),
        unlearn_method=f'{base_method}_CMF_RemoveFC',
        unlearn_class=[forget_class],
        batch_size=128, test_batch_size=256, lr=lr,
        momentum=0.9, weight_decay=5e-4, epochs_or_steps=epochs,
        seed=seed,
        num_retain_samples=len(retain_train_idx),
        num_forget_samples=len(forget_train_idx),
        grad_norm_clip=1.0,
        SVD_alpha_r=1000, SVD_alpha_f=30,
        SVD_samples=900, SVD_max_patches=10000,
        freeze_except_last=False,
        scrub_del_bsz=64, scrub_sgda_bsz=64, scrub_msteps=2, scrub_epochs=epochs,
        salun_threshold=0.5,
        tarun_impair_lr=lr, tarun_samples_per_class=1000,
        dry_run=TEST_MODE, no_cuda=False, no_mps=True, gamma=0.5,
        data_path='/kaggle/working/data', remove_FC=True,
        CMFClassifier=True, CMF_momentum=0.9, pretrained=False, temperature=1.0,
        prob_batch_size=256, lp_every=0, mean_source=mean_source,
        repo_commit=REPO_COMMIT, test_mode=TEST_MODE,
    )

@torch.no_grad()
def eval_acc(model, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        correct += (model(x.to(device)).argmax(1).cpu() == y).sum().item()
        total   += y.size(0)
    return 100.0 * correct / max(total, 1)

@torch.no_grad()
def cmf_extract_features(model, loader):
    model.eval()
    feats, labs = [], []
    for x, y in loader:
        x = x.to(device)
        f = model.extract_features(x)
        z = model._preprocess_feats_for_cmf(f)
        feats.append(z.cpu()); labs.append(y)
    return torch.cat(feats), torch.cat(labs)

def run_probe_cmf(model, train_retain_ldr, train_forget_ldr,
                  test_retain_ldr, test_forget_ldr, n_epochs=50):
    Xtr, ytr = cmf_extract_features(model, train_retain_ldr)
    Xfg, yfg = cmf_extract_features(model, train_forget_ldr)
    Xall = torch.cat([Xtr, Xfg])
    yall = torch.cat([ytr, yfg])
    head = nn.Linear(Xall.size(1), NUM_CLASSES).to(device)
    opt  = optim.SGD(head.parameters(), lr=1e-2, momentum=0.9)
    ldr  = torch.utils.data.DataLoader(
               torch.utils.data.TensorDataset(Xall, yall), batch_size=256, shuffle=True)
    for _ in range(n_epochs):
        head.train()
        for bx, by in ldr:
            opt.zero_grad()
            F.cross_entropy(head(bx.to(device)), by.to(device)).backward()
            opt.step()
    head.eval()
    with torch.no_grad():
        Xte_r, yte_r = cmf_extract_features(model, test_retain_ldr)
        Xte_f, yte_f = cmf_extract_features(model, test_forget_ldr)
        ret_acc = (head(Xte_r.to(device)).argmax(1).cpu() == yte_r).float().mean().item() * 100
        fgt_acc = (head(Xte_f.to(device)).argmax(1).cpu() == yte_f).float().mean().item() * 100
    return ret_acc, fgt_acc

def run_ncc_cmf(model, train_retain_ldr, train_forget_ldr,
                test_retain_ldr, test_forget_ldr):
    Xtr, ytr = cmf_extract_features(model, train_retain_ldr)
    Xfg, yfg = cmf_extract_features(model, train_forget_ldr)
    Xall = torch.cat([Xtr, Xfg])
    yall = torch.cat([ytr, yfg])
    means = []
    for c in range(NUM_CLASSES):
        mask = (yall == c)
        mu = Xall[mask].mean(0) if mask.any() else torch.zeros(Xall.size(1))
        means.append(mu)
    M = torch.stack(means)
    Xte_r, yte_r = cmf_extract_features(model, test_retain_ldr)
    Xte_f, yte_f = cmf_extract_features(model, test_forget_ldr)
    ret_pred = torch.cdist(Xte_r.unsqueeze(0), M.unsqueeze(0)).squeeze(0).argmin(1)
    fgt_pred = torch.cdist(Xte_f.unsqueeze(0), M.unsqueeze(0)).squeeze(0).argmin(1)
    return ((ret_pred == yte_r).float().mean().item() * 100,
            (fgt_pred == yte_f).float().mean().item() * 100)

def eval_cmf_three_metrics(model,
                           test_retain_ldr, test_forget_ldr,
                           train_retain_eval_ldr, train_forget_eval_ldr):
    out_ret = eval_acc(model, test_retain_ldr)
    out_fgt = eval_acc(model, test_forget_ldr)
    lp_ret, lp_fgt   = run_probe_cmf(model, train_retain_eval_ldr, train_forget_eval_ldr,
                                      test_retain_ldr, test_forget_ldr)
    ncc_ret, ncc_fgt = run_ncc_cmf(model, train_retain_eval_ldr, train_forget_eval_ldr,
                                    test_retain_ldr, test_forget_ldr)
    return {'output_retain_acc': out_ret, 'output_forget_acc': out_fgt,
            'probe_retain_acc':  lp_ret,  'probe_forget_acc':  lp_fgt,
            'ncc_retain_acc':    ncc_ret, 'ncc_forget_acc':    ncc_fgt}

print('Helpers ready.')

Helpers ready.


In [7]:
# ── Load theta_o ────────────────────────────────────────────────────────────
theta_o_candidates = [
    f'{CKPT_ROOT_NB1}/pre_train/theta_o_seed{THETA_O_SEED}.pt',
    f'{CKPT_ROOT_NB1}/pre_train/cifar10_resnet18_full.pt',
    './notebooks/Result_nb1/checkpoints/regun/pre_train/cifar10_resnet18_full.pt',
]
theta_o_path = None
for _p in theta_o_candidates:
    if os.path.exists(_p):
        theta_o_path = _p; break
if theta_o_path is None and os.path.exists('./checkpoints/pre_train/cifar10_resnet18_full.pt'):
    theta_o_path = './checkpoints/pre_train/cifar10_resnet18_full.pt'

if theta_o_path and os.path.exists(theta_o_path):
    print(f'Loading theta_o from {theta_o_path}')
    ck_o = torch.load(theta_o_path, map_location=device)
    theta_o_state = ck_o.get('model_state_dict', ck_o)
else:
    print('WARNING: theta_o checkpoint not found. Creating initialized backbone state.')
    from models.resnet import ResNet18
    _init_m = ResNet18(num_classes=NUM_CLASSES, dataset=DATASET)
    theta_o_state = _init_m.state_dict()

results_4a = []

for forget_class in FORGET_CLASSES:
    for seed in SEEDS:
        forget_train_idx = TRAIN_CLASS_IDX[forget_class]
        retain_train_idx = [
            i for c in range(NUM_CLASSES)
            if c != forget_class
            for i in TRAIN_CLASS_IDX[c]
        ]
        retain_loader = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train, retain_train_idx),
            batch_size=128, shuffle=True, num_workers=2)
        forget_loader = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train, forget_train_idx),
            batch_size=128, shuffle=True, num_workers=2)
        full_train_loader = torch.utils.data.DataLoader(
            full_train, batch_size=256, shuffle=False, num_workers=2)
        train_retain_eval_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train_eval, retain_train_idx),
            batch_size=256, shuffle=False, num_workers=2)
        train_forget_eval_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(full_train_eval, forget_train_idx),
            batch_size=256, shuffle=False, num_workers=2)
        test_forget_idx = TEST_CLASS_IDX[forget_class]
        test_retain_idx = [
            i for c in range(NUM_CLASSES)
            if c != forget_class
            for i in TEST_CLASS_IDX[c]
        ]
        test_forget_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(test_set, test_forget_idx),
            batch_size=256, shuffle=False, num_workers=2)
        test_retain_ldr = torch.utils.data.DataLoader(
            torch.utils.data.Subset(test_set, test_retain_idx),
            batch_size=256, shuffle=False, num_workers=2)
        test_loader_full = torch.utils.data.DataLoader(
            test_set, batch_size=256, shuffle=False, num_workers=2)

        for base_method in BASE_METHODS:
            for mean_source in MEAN_SOURCES:
                cmf_epochs = CMF_EPOCHS_BY_METHOD.get(base_method, 3)
                lr    = CMF_LR.get(base_method, 1e-3)
                batch = CMF_BATCH.get(base_method, CMF_BATCH_DEFAULT)
                tag   = f'{base_method}_cmf_static_{mean_source}_class{forget_class}_seed{seed}'
                if TEST_MODE: tag += '_testmode'
                ckpt_path = f'{CKPT_ROOT}/cmf_static/{tag}.pt'

                if os.path.exists(ckpt_path):
                    print(f'[{tag}] exists — loading cached result.')
                    ck = torch.load(ckpt_path, map_location=device)
                    results_4a.append(ck['metrics'])
                    continue

                print(f'\n[{tag}] Running cmf_static ({base_method}) for {cmf_epochs} epochs...')
                torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)

                args = make_cmf_args(base_method, lr, cmf_epochs, mean_source,
                                     forget_class, forget_train_idx, retain_train_idx,
                                     seed=seed)
                args.batch_size = batch
                if base_method == 'scrub':
                    args.scrub_del_bsz  = 64
                    args.scrub_sgda_bsz = 64

                model = build_cmf_model(args)
                model.encoder.load_state_dict(theta_o_state, strict=False)

                mean_loader = full_train_loader if mean_source == 'train' else retain_loader
                model.eval()
                model.recompute_cmf(mean_loader, device=device)

                t0 = time.time()
                try:
                    model = run_cmf_static(
                        base_method=base_method,
                        args=args,
                        model=model,
                        device=device,
                        retain_loader=retain_loader,
                        forget_loader=forget_loader,
                        train_loader=mean_loader,
                        test_loader=test_loader_full,
                        epochs=cmf_epochs,
                        test_forget_loader=test_forget_ldr,
                        train_dataset=full_train,
                        val_index=retain_train_idx,
                    )
                except Exception as e:
                    print(f'ERROR running {base_method}: {e}')
                    traceback.print_exc()
                    continue
                wall_min = (time.time() - t0) / 60

                model.eval()
                model.recompute_cmf(mean_loader, device=device)

                metrics = eval_cmf_three_metrics(
                    model,
                    test_retain_ldr, test_forget_ldr,
                    train_retain_eval_ldr, train_forget_eval_ldr
                )
                metrics.update({
                    'method': base_method, 'mean_source': mean_source,
                    'forget_class': forget_class, 'seed': seed, 'stage': STAGE,
                    'cmf_epochs': cmf_epochs, 'lr': lr, 'batch': batch,
                    'wall_clock_minutes': wall_min,
                    'n_forget_train': len(forget_train_idx),
                    'n_retain_train': len(retain_train_idx),
                    'protocol': 'whole_class_single',
                })

                torch.save({
                    'model_state_dict': model.state_dict(),
                    'config': {
                        'base_method': base_method, 'mean_source': mean_source,
                        'dataset': DATASET, 'arch': ARCH, 'num_classes': NUM_CLASSES,
                        'forget_class': forget_class, 'seed': seed,
                        'cmf_epochs': cmf_epochs, 'lr': lr, 'batch': batch,
                        'protocol': 'whole_class_single',
                        'repo_commit': REPO_COMMIT, 'test_mode': TEST_MODE,
                    },
                    'seed': seed, 'metrics': metrics,
                }, ckpt_path)
                print(f'  Saved {ckpt_path}')
                print(f'  out  R={metrics["output_retain_acc"]:.2f}%  F={metrics["output_forget_acc"]:.2f}%')
                print(f'  probe R={metrics["probe_retain_acc"]:.2f}%  F={metrics["probe_forget_acc"]:.2f}%')
                print(f'  ncc  R={metrics["ncc_retain_acc"]:.2f}%  F={metrics["ncc_forget_acc"]:.2f}%')
                results_4a.append(metrics)

df_4a = pd.DataFrame(results_4a)
csv_path = f'{CKPT_ROOT}/results_4a_cmf_static.csv'
df_4a.to_csv(csv_path, index=False)
print(f'\nResults saved to {csv_path}')
if not df_4a.empty:
    metric_cols = ['output_retain_acc','output_forget_acc',
                   'probe_retain_acc','probe_forget_acc','ncc_retain_acc','ncc_forget_acc']
    print('\n=== Summary ===')
    print(df_4a.groupby('method')[metric_cols].mean().round(2).to_string())

Loading theta_o from /kaggle/input/datasets/btk23021592/cmf-notebook1/checkpoints/cmf_benchmark/pre_train/theta_o_seed0.pt

[grad_ascent_descent_cmf_static_train_class0_seed0] Running cmf_static (grad_ascent_descent) for 3 epochs...
Removing FC layer from the encoder and using CMF classifier.
CMF_momentum: 0.9
[ModelModule] arch=resnet18, is_vit=False, remove_FC=True, feature_dim=512
[Before Unlearning] Evaluating CMF model
args.CMF_momentum = 0.9
model.args.CMF_momentum = 0.9
Test Set (Epoch 0): Average loss: 1.4400, Accuracy: 9445/10000 (94%)
------------------------------
Test Set (Epoch 0) Confusion Matrix 
[[962.   1.  12.   0.   1.   2.   1.   1.  18.   2.]
 [  5. 974.   0.   0.   0.   0.   0.   1.   4.  16.]
 [ 13.   0. 917.  11.  19.  13.  19.   4.   4.   0.]
 [  4.   1.  17. 879.  13.  59.  15.   4.   5.   3.]
 [  2.   0.   7.  15. 957.   2.  10.   6.   1.   0.]
 [  3.   1.  10.  54.  12. 907.   2.   8.   1.   2.]
 [  6.   1.   8.  13.   3.   3. 965.   1.   0.   0.]
 [  7.   2

/usr/local/lib/python3.12/dist-packages/torch/nn/_reduction.py:60: UserWarning: size_average and reduce args will be deprecated, please use reduction='sum' instead.
  return get_enum(legacy_get_string(size_average, reduce, emit_warning))


 * Acc@1 99.040 
Test Set (Epoch 1): Average loss: 1.4794, Accuracy: 8925/10000 (89%)
------------------------------
Test Set (Epoch 1) Confusion Matrix 
[[538.  37. 176.  54.  43.  10.  10.  19.  74.  39.]
 [  0. 984.   1.   1.   0.   1.   0.   0.   2.  11.]
 [  1.   2. 894.  18.  36.  12.  22.  11.   4.   0.]
 [  1.   3.  12. 902.  18.  37.  17.   6.   3.   1.]
 [  0.   0.   7.  11. 958.   1.  10.  12.   1.   0.]
 [  0.   1.   6. 105.  15. 847.  12.  12.   0.   2.]
 [  0.   1.  11.  19.   6.   1. 961.   0.   0.   1.]
 [  0.   1.   5.  18.  12.  11.   1. 950.   2.   0.]
 [  1.  15.   3.   6.   2.   1.   3.   1. 964.   4.]
 [  1.  60.   1.   6.   0.   0.   0.   0.   5. 927.]]
------------------------------
Test Set (Epoch 1) Class Wise ACC 
{0: np.float64(53.800000000000004), 1: np.float64(98.4), 2: np.float64(89.4), 3: np.float64(90.2), 4: np.float64(95.8), 5: np.float64(84.7), 6: np.float64(96.1), 7: np.float64(95.0), 8: np.float64(96.39999999999999), 9: np.float64(92.7)}
-----------

/usr/local/lib/python3.12/dist-packages/torch/nn/_reduction.py:60: UserWarning: size_average and reduce args will be deprecated, please use reduction='sum' instead.
  return get_enum(legacy_get_string(size_average, reduce, emit_warning))


 * Acc@1 98.413 
Test Set (Epoch 2): Average loss: 1.4986, Accuracy: 8619/10000 (86%)
------------------------------
Test Set (Epoch 2) Confusion Matrix 
[[297.  26. 228.  98.  56.   6.  28.  19. 184.  58.]
 [  0. 973.   2.   4.   0.   0.   2.   0.   5.  14.]
 [  0.   0. 897.  24.  22.  20.  25.   5.   7.   0.]
 [  0.   0.  24. 871.  10.  63.  17.   7.   6.   2.]
 [  0.   0.  10.  18. 948.   5.  11.   7.   1.   0.]
 [  0.   0.   4.  81.  20. 886.   4.   5.   0.   0.]
 [  0.   1.  19.  21.   2.   2. 953.   1.   0.   1.]
 [  0.   2.  10.  24.  54.  19.   0. 888.   2.   1.]
 [  0.  12.   4.   7.   3.   0.   2.   2. 964.   6.]
 [  0.  38.   1.   7.   0.   0.   1.   0.  11. 942.]]
------------------------------
Test Set (Epoch 2) Class Wise ACC 
{0: np.float64(29.7), 1: np.float64(97.3), 2: np.float64(89.7), 3: np.float64(87.1), 4: np.float64(94.8), 5: np.float64(88.6), 6: np.float64(95.3), 7: np.float64(88.8), 8: np.float64(96.39999999999999), 9: np.float64(94.19999999999999)}
------------

...

 * Acc@1 98.858 
Test Set (Epoch 1): Average loss: 1.4764, Accuracy: 8777/10000 (88%)
------------------------------
Test Set (Epoch 1) Confusion Matrix 
[[920.   8.  23.   9.   6.   1.   5.   2.  26.   0.]
 [  3. 989.   0.   0.   0.   0.   1.   0.   6.   1.]
 [ 24.   0. 919.  11.  23.   8.  10.   3.   2.   0.]
 [  3.   2.  39. 841.  12.  79.  14.   6.   4.   0.]
 [  6.   0.  17.  11. 945.   8.  11.   2.   0.   0.]
 [  6.   2.  25.  52.  15. 884.   4.  10.   2.   0.]
 [  1.   0.  33.  23.   3.   5. 935.   0.   0.   0.]
 [  9.   2.  16.  15.  34.  22.   3. 898.   1.   0.]
 [ 20.  13.   9.   2.   0.   0.   3.   0. 953.   0.]
 [ 28. 394.   9.  14.   0.   1.   5.   3.  53. 493.]]
------------------------------
Test Set (Epoch 1) Class Wise ACC 
{0: np.float64(92.0), 1: np.float64(98.9), 2: np.float64(91.9), 3: np.float64(84.1), 4: np.float64(94.5), 5: np.float64(88.4), 6: np.float64(93.5), 7: np.float64(89.8), 8: np.float64(95.3), 9: np.float64(49.3)}
------------------------------
Test Se

/usr/local/lib/python3.12/dist-packages/torch/nn/_reduction.py:60: UserWarning: size_average and reduce args will be deprecated, please use reduction='sum' instead.
  return get_enum(legacy_get_string(size_average, reduce, emit_warning))


 * Acc@1 98.484 
Test Set (Epoch 2): Average loss: 1.4909, Accuracy: 8590/10000 (86%)
------------------------------
Test Set (Epoch 2) Confusion Matrix 
[[913.   2.  14.   8.   3.   3.   8.   5.  44.   0.]
 [  6. 959.   0.   1.   0.   0.   2.   2.  17.  13.]
 [ 20.   0. 855.  30.  29.  22.  31.   7.   6.   0.]
 [  6.   0.  12. 838.  11.  85.  26.  11.  11.   0.]
 [  5.   0.  11.  23. 913.  18.  14.  12.   4.   0.]
 [  7.   0.   8.  75.  14. 871.  12.  13.   0.   0.]
 [  3.   0.   5.  18.   4.   7. 962.   0.   1.   0.]
 [  5.   0.   2.  18.  12.  17.   2. 943.   1.   0.]
 [ 10.   1.   2.   6.   0.   2.   0.   0. 976.   3.]
 [ 81. 330.   4.  30.   2.   2.  13.  36. 142. 360.]]
------------------------------
Test Set (Epoch 2) Class Wise ACC 
{0: np.float64(91.3), 1: np.float64(95.89999999999999), 2: np.float64(85.5), 3: np.float64(83.8), 4: np.float64(91.3), 5: np.float64(87.1), 6: np.float64(96.2), 7: np.float64(94.3), 8: np.float64(97.6), 9: np.float64(36.0)}
-------------------------

/usr/local/lib/python3.12/dist-packages/torch/nn/_reduction.py:60: UserWarning: size_average and reduce args will be deprecated, please use reduction='sum' instead.
  return get_enum(legacy_get_string(size_average, reduce, emit_warning))


 * Acc@1 98.144 
Test Set (Epoch 3): Average loss: 1.4945, Accuracy: 8512/10000 (85%)
------------------------------
Test Set (Epoch 3) Confusion Matrix 
[[902.   8.  20.   0.   5.   3.   3.   2.  52.   5.]
 [  5. 973.   0.   0.   1.   0.   0.   0.  10.  11.]
 [ 13.   1. 917.  22.  19.   7.  12.   1.   8.   0.]
 [ 18.   4.  35. 838.  15.  67.  11.   2.  10.   0.]
 [  3.   1.  18.  25. 930.   9.   5.   5.   4.   0.]
 [  5.   4.  26.  67.  18. 865.   7.   6.   2.   0.]
 [  4.   1.  20.  28.   9.   4. 930.   2.   2.   0.]
 [ 14.   1.   9.  23.  33.  26.   3. 886.   5.   0.]
 [  8.   4.   5.   0.   0.   0.   0.   0. 975.   8.]
 [ 67. 405.   5.   9.   0.   0.   3.   1. 214. 296.]]
------------------------------
Test Set (Epoch 3) Class Wise ACC 
{0: np.float64(90.2), 1: np.float64(97.3), 2: np.float64(91.7), 3: np.float64(83.8), 4: np.float64(93.0), 5: np.float64(86.5), 6: np.float64(93.0), 7: np.float64(88.6), 8: np.float64(97.5), 9: np.float64(29.599999999999998)}
------------------------